In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from dotenv import load_dotenv
import os
import torch
import json
import time

# Change working directory to DRAFT
os.chdir('DRAFT')
new_directory = os.getcwd()
print(f"Working directory: {new_directory}")

load_dotenv()
HUGGINGFACE_TOKEN = os.getenv("HUGGINGFACE")

Working directory: /research/phd/phd2k22/cse/rudra.dhar/DRAFT


In [2]:
# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

cache_dir - "../cache"
model_name = "google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir, token=HUGGINGFACE_TOKEN)
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=cache_dir, token=HUGGINGFACE_TOKEN, device_map="auto")

# Move the model to the chosen device
model.to(device)

# Set the model to evaluation mode
model.eval()

Using device: cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
          (position_embedding): Embedding(4096, 1152)
        )
        (encoder): SiglipEncoder(
          (layers): ModuleList(
            (0-26): 27 x SiglipEncoderLayer(
              (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (self_attn): SiglipAttention(
                (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
              )
              (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwi

In [5]:
"""
Define helper functions
"""
def load_jsonl(file_path):
    """Load a JSONL file and return list of dicts."""
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
            
        
    return data

def save_jsonl(data, file_path):
    """Append list of dicts to a JSONL file."""
    with open(file_path, "a", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
        
    

def generate_response(model, tokenizer, messages, device, max_new_tokens=500):
    """Generate model response given messages."""
    formatted_chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Tokenize
    inputs = tokenizer(formatted_chat, return_tensors="pt").to(device)
    input_length = inputs["input_ids"].shape[1]
    # Generate
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    generated_ids = outputs[0][input_length:]  # slice only new tokens
    response = tokenizer.decode(generated_ids, skip_special_tokens=True) # Decode only the generated text
    generated_tokens = generated_ids.shape[0] # Number of generated tokens
    return response, generated_tokens


def extract_context(entry):
    """Extract context string from one JSONL entry."""
    return entry["Anchor"]["Context"]

def extract_title(entry):
    """Extract title from one JSONL entry."""
    return entry["Anchor"]["Title"]

def extract_primary_key(entry):
    """Extract primary key from one JSONL entry."""
    return entry["Anchor"]["PrimaryKey"]

def context_formator(context):
    messages = [
        {
            "role": "system", 
            "content": "You are an expert software architect responsible for maintaining and thoroughly documenting all architectural decisions. You are writing an Architectural Decision Record for a software. Give a ## Decision corresponding to the ## Context provided by the User. Provide only the Decision in about 2-400 words. Do not add any explanations, introductions, or additional responses."
        },
        {
            "role": "user",
            "content": f"## Context: {context}"
        }
    ]
    return messages

def title_formator(title):
    messages = [
        {
            "role": "system", 
            "content": "You are an expert software architect responsible for maintaining and thoroughly documenting all architectural decisions. You are writing an Architectural Decision Record for a software. Write the ADR corresponding to the ADR Title provided by the User. Provide only the ADR content in about 10-800 words. Do not add any additional responses—only the ADR content."
        },
        {
            "role": "user",
            "content": f"# {title}"
        }
    ]
    return messages


### Context to Decision

In [ ]:
input_file = "Retrieval/CDtest.jsonl"
output_file = "Prompting/Results/gemma-3-4b-it-CDtest-results.jsonl"
entries = load_jsonl(input_file)

results = []

# Iterate over entries
for i, entry in enumerate(entries[:3]): # limit to first 3 for demo
    primary_key = extract_primary_key(entry)
    context = extract_context(entry)
    messages = context_formator(context)

    start_time = time.time()
    response, gen_tokens = generate_response(model, tokenizer, messages, device)
    elapsed = time.time() - start_time


    result = {
    "PrimaryKey": primary_key,
    "Decision": response,
    "GeneratedTokens": gen_tokens,
    "Time": elapsed
    }
    results.append(result)

# Save all results to output JSONL
save_jsonl(results, output_file)
print(f"Results saved to {output_file}")

Results saved to Prompting/Results/gemma-3-4b-it-CDtest-results.jsonl


### Title to Body

In [ ]:
input_file = "Retrieval/TBtest.jsonl"
output_file = "Prompting/Results/gemma-3-4b-it-TBtest-results.jsonl"
entries = load_jsonl(input_file)

results = []

# Iterate over entries
for i, entry in enumerate(entries[:3]): # limit to first 3 for demo
    primary_key = extract_primary_key(entry)
    title = extract_title(entry)
    messages = title_formator(title)

    start_time = time.time()
    response, gen_tokens = generate_response(model, tokenizer, messages, device, max_new_tokens=1000)
    elapsed = time.time() - start_time

    result = {
    "PrimaryKey": primary_key,
    "Body": response,
    "GeneratedTokens": gen_tokens,
    "Time": elapsed
    }
    results.append(result)

# Save all results to output JSONL
save_jsonl(results, output_file)
print(f"Results saved to {output_file}")

Results saved to Prompting/Results/gemma-3-4b-it-TBtest-results.jsonl
